# Walmart Sales Forecasting — Inference & Submission

**მიზანი:** საუკეთესო მოდელით (**TimesFM**, zero-shot) test set-ზე პროგნოზი და Kaggle submission-ის მომზადება.

## 10 მოდელის საბოლოო შედარება

| Rank | Model | Val WMAE | Category | Owner |
|------|-------|----------|----------|-------|
| 1 | **TimesFM (zero-shot)** | **1309.76** | Foundation | giga |
| 2 | XGBoost | 1321.74 | Tree-based | Zaqaria |
| 3 | Prophet (full population) | 1331.57 | Classical | giga |
| 4 | Ensemble (XGBoost + Prophet, weighted blend) | 1332.23 | Ensemble | Team |
| 5 | LightGBM | 1342.90 | Tree-based | giga |
| 6 | N-BEATS | 1378.04 | DL (feed-forward) | Zaqaria |
| 7 | PatchTST | 1420.44 | DL (transformer) | Zaqaria |
| 8 | DLinear | 1494.80 | DL (linear) | giga |
| 9 | TFT | 1538.41 | DL (LSTM + attention) | Zaqaria |
| 10 | SARIMA | 7012.96 | Classical | Zaqaria |

**გამარჯვებული:** **TimesFM** (zero-shot) — მიუხედავად იმისა, რომ Walmart-ის მონაცემები არასდროს უნახავს, TimesFM მაინც წინ არის, თუმცა მინიმალური მარჟით (მხოლოდ 11.98 WMAE-ით უსწრებს გაუმჯობესებულ XGBoost-ს — 1309.76 vs 1321.74). ეს notebook:
1. Load-ს pretrained TimesFM მოდელს (`google/timesfm-2.5-200m-pytorch`) HuggingFace-იდან — **fine-tuning არ სჭირდება**
2. აწარმოებს zero-shot პროგნოზებს **raw** train.csv-ის ისტორიაზე (context) თითოეული (Store, Dept) სერიისთვის, raw test.csv-ის თარიღებზე (horizon)
3. ქმნის Kaggle submission CSV-ს (`Id, Weekly_Sales` format)

## 1. Setup

In [ ]:
!pip install timesfm pandas numpy --quiet

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import torch
import timesfm

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/walmart'
DATA_DIR = f'{PROJECT_DIR}/data'
SUBMISSIONS_DIR = f'{PROJECT_DIR}/submissions'

import os
os.makedirs(SUBMISSIONS_DIR, exist_ok=True)

print(f"Data:        {DATA_DIR}")
print(f"Submissions: {SUBMISSIONS_DIR}")

## 2. TimesFM მოდელის ჩატვირთვა

TimesFM Google-ის HuggingFace repo-იდან იტვირთება — pretrained, **fine-tuning არ სჭირდება**. `model_experiment_TimesFM.ipynb`-ში best config იყო **256-კვირიანი context**, ამიტომ იმავეს ვიყენებთ აქაც. Kaggle test set 39 კვირას მოიცავს, ამიტომ `max_horizon`-ს 39-მდე ვზრდით (val ექსპერიმენტში 12 იყო, რადგან იქ val window მხოლოდ 12 კვირა იყო).

In [ ]:
BEST_CONTEXT = 256   # model_experiment_TimesFM.ipynb-ის საუკეთესო კონფიგი (val WMAE 1309.76)
MAX_HORIZON = 39      # Kaggle test set-ის სრული ჰორიზონტი (2012-11 -> 2013-07)

tfm = timesfm.TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch")

tfm.compile(
    timesfm.ForecastConfig(
        max_context=512,
        max_horizon=MAX_HORIZON,
        normalize_inputs=True,
        use_continuous_quantile_head=True,
        force_flip_invariance=True,
        infer_is_positive=True,
        fix_quantile_crossing=True,
    )
)

print("TimesFM ჩაიტვირთა წარმატებით")

## 3. Raw train + test მონაცემების ჩატვირთვა

TimesFM-ს context-ისთვის სჭირდება ისტორიული გაყიდვები (`train.csv`), horizon-ისთვის კი test-ის თარიღები (`test.csv`). Store/features metadata (CPI, MarkDown და ა.შ.) TimesFM-ს არ სჭირდება — ის მხოლოდ `y`-ის (Weekly_Sales) ისტორიაზეა დამოკიდებული.

In [ ]:
train_raw = pd.read_csv(f'{DATA_DIR}/train.csv.zip')
test_raw = pd.read_csv(f'{DATA_DIR}/test.csv.zip')

train_raw['Date'] = pd.to_datetime(train_raw['Date'])
test_raw['Date'] = pd.to_datetime(test_raw['Date'])

print(f"Train shape: {train_raw.shape}")
print(f"Test shape:  {test_raw.shape}")
print(f"Train date range: {train_raw['Date'].min()} -> {train_raw['Date'].max()}")
print(f"Test date range:  {test_raw['Date'].min()} -> {test_raw['Date'].max()}")

## 4. Long-format ტრანსფორმაცია

იგივე ფორმატი, რაც `model_experiment_TimesFM.ipynb`-ში — `unique_id` (Store_Dept), `ds` (თარიღი), `y` (target).

In [ ]:
def to_timesfm_format(df, has_target=True):
    df = df.copy()
    df['unique_id'] = df['Store'].astype(str) + '_' + df['Dept'].astype(str)
    df = df.rename(columns={'Date': 'ds'})
    if has_target:
        df = df.rename(columns={'Weekly_Sales': 'y'})
        return df[['unique_id', 'ds', 'y']]
    return df[['unique_id', 'ds']]


train_tfm = to_timesfm_format(train_raw, has_target=True).sort_values(['unique_id', 'ds']).reset_index(drop=True)
test_tfm = to_timesfm_format(test_raw, has_target=False).sort_values(['unique_id', 'ds']).reset_index(drop=True)

train_ids = set(train_tfm['unique_id'].unique())
test_ids = set(test_tfm['unique_id'].unique())
new_ids = test_ids - train_ids

print(f"Train series: {len(train_ids)}")
print(f"Test series:  {len(test_ids)}")
print(f"Test-ში ახალი (train-ში არარსებული) სერიები: {len(new_ids)}")

## 5. Inference — zero-shot პროგნოზები raw test-ზე

თითოეული (Store, Dept) სერიისთვის: ბოლო 256 კვირა `train`-იდან context-ად, `test`-ის თარიღები horizon-ად. სერიები, რომლებიც test-ში ახლად ჩნდება (`new_ids`) — history არ აქვთ, ამიტომ TimesFM-ს ვერ მივაწვდით context-ს; ასეთი რიგები fallback მედიანით ივსება მე-6 სექციაში.

In [ ]:
def forecast_all_series(tfm_model, train_df, test_df, context_len):
    forecasts_list = []
    unique_ids = test_df['unique_id'].unique()

    for uid in unique_ids:
        series_hist = train_df[train_df['unique_id'] == uid].sort_values('ds')
        series_future_dates = test_df[test_df['unique_id'] == uid].sort_values('ds')['ds'].values
        horizon = len(series_future_dates)

        if len(series_hist) == 0 or horizon == 0:
            continue

        context = series_hist['y'].values[-context_len:].astype(np.float32)

        try:
            point_forecast, quantile_forecast = tfm_model.forecast(
                horizon=horizon,
                inputs=[context],
            )

            for date, pred in zip(series_future_dates, point_forecast[0][:horizon]):
                forecasts_list.append({
                    'unique_id': uid,
                    'ds': date,
                    'TimesFM': float(pred)
                })
        except Exception as e:
            continue

    return pd.DataFrame(forecasts_list)


print(f"Zero-shot ინფერენსი {test_tfm['unique_id'].nunique()} სერიაზე (context={BEST_CONTEXT})...")
forecasts = forecast_all_series(tfm, train_tfm, test_tfm, BEST_CONTEXT)

print(f"\nProgress: {forecasts['unique_id'].nunique()} / {test_tfm['unique_id'].nunique()} სერია დაფარულია")
print(f"Forecast rows: {len(forecasts):,}")

## 6. პროგნოზების შერწყმა raw test-თან + fallback

`forecasts`-ს ვაერთებთ `test_raw`-თან `(unique_id, ds)`-ზე. სერიები, რომლებზეც zero-shot ვერ იმუშავა (ახალი Store/Dept კომბინაცია ან TimesFM-ის შეცდომა) — fallback-ად გლობალურ მედიანას იღებენ, რომ submission-ში ცარიელი მნიშვნელობა არ დარჩეს.

In [ ]:
test_tfm_ids = test_raw.copy()
test_tfm_ids['unique_id'] = test_tfm_ids['Store'].astype(str) + '_' + test_tfm_ids['Dept'].astype(str)
test_tfm_ids = test_tfm_ids.rename(columns={'Date': 'ds'})

merged = test_tfm_ids.merge(forecasts, on=['unique_id', 'ds'], how='left')

n_missing = merged['TimesFM'].isna().sum()
print(f"Test rows:              {len(merged):,}")
print(f"Rows covered by TimesFM: {(~merged['TimesFM'].isna()).sum():,}")
print(f"Rows needing fallback:   {n_missing:,}")

if n_missing > 0:
    fallback_median = train_raw['Weekly_Sales'].median()
    merged['TimesFM'] = merged['TimesFM'].fillna(fallback_median)
    print(f"Fallback (train-wide median) გამოყენებულია: {fallback_median:.2f}")

test_predictions = merged['TimesFM'].values

print(f"\nPrediction stats:")
print(f"  Min:    {test_predictions.min():.2f}")
print(f"  Max:    {test_predictions.max():.2f}")
print(f"  Mean:   {test_predictions.mean():.2f}")
print(f"  Median: {np.median(test_predictions):.2f}")

## 7. Kaggle Submission Format

In [ ]:
submission = pd.DataFrame({
    'Id': (
        test_raw['Store'].astype(str) + '_' +
        test_raw['Dept'].astype(str) + '_' +
        test_raw['Date'].dt.strftime('%Y-%m-%d')
    ),
    'Weekly_Sales': test_predictions
})

# უარყოფითი პროგნოზები 0-ით ვცვლით (გაყიდვები ვერ იქნება ნეგატიური)
n_negative = (submission['Weekly_Sales'] < 0).sum()
if n_negative > 0:
    print(f"Negative predictions clipped: {n_negative}")
    submission['Weekly_Sales'] = submission['Weekly_Sales'].clip(lower=0)

print(f"\nSubmission shape: {submission.shape}")
print(f"\nSubmission head:")
print(submission.head(10).to_string(index=False))

## 8. Submission CSV-ის შენახვა

In [ ]:
from datetime import datetime

# ვერსია timestamp-ით
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
submission_path = f'{SUBMISSIONS_DIR}/timesfm_submission_{timestamp}.csv'
submission.to_csv(submission_path, index=False)

# ასევე ერთი "latest" ვერსია
latest_path = f'{SUBMISSIONS_DIR}/submission_latest.csv'
submission.to_csv(latest_path, index=False)

print(f"Submission შენახულია:")
print(f"  {submission_path}")
print(f"  {latest_path}")
print(f"\nRows: {len(submission):,}")
print(f"File size: {os.path.getsize(submission_path) / 1024:.1f} KB")

## 9. შეჯამება

### რა გავაკეთეთ

1. **Load** — pretrained TimesFM (`google/timesfm-2.5-200m-pytorch`), zero-shot, fine-tuning გარეშე (val WMAE 1309.76 — 10 მოდელს შორის საუკეთესო)
2. **Inference** — თითოეული (Store, Dept) სერიისთვის ბოლო 256 კვირა `train.csv`-იდან, როგორც context, `test.csv`-ის თარიღები, როგორც horizon
3. **Fallback** — history-ის არმქონე ან ვერ-დაფარულ სერიებზე გლობალური მედიანა
4. **Format** — Kaggle-ის სპეციფიკური `Id, Weekly_Sales` submission
5. **Save** — timestamped + latest CSV Drive-ზე

### რატომ TimesFM და არა XGBoost

XGBoost-ის feature engineering-ის გაუმჯობესების შემდეგ (Store×Dept lag/rolling ისტორია + Dept×Week სეზონური საშუალო) მისი val WMAE **1321.74**-მდე ჩამოვიდა — ეს ძალიან ახლოსაა TimesFM-თან (1309.76) და უკვე სჯობს Prophet-ს (1331.57), Ensemble-ს (1332.23), LightGBM-ს (1342.90) და N-BEATS-ს (1378.04). სხვაობა TimesFM-თან მხოლოდ **11.98 WMAE-ია** — პრაქტიკულად noise-ის ფარგლებშია, მაგრამ ამ notebook-ის დაწერის მომენტისთვის TimesFM ისევ ოფიციალური winner-ია, ამიტომ inference ამ მოდელს იყენებს. თუ მომავალში XGBoost-ის კიდევ ერთი გაუმჯობესება ამ 12-ერთეულიან სხვაობას გადალახავს, winner-ი და ეს notebook განახლდება.

### Kaggle Submission

| Score Type | WMAE |
|-----------|------|
| Private Score | *(TBD — submission-ის შემდეგ ივსება)* |
| Public Score | *(TBD — submission-ის შემდეგ ივსება)* |

### ლინკები

- **MLflow (DagsHub):** https://dagshub.com/zberi23/walmart-forecasting.mlflow
- **WandB (Zaqaria):** https://wandb.ai/zberi23_ml/walmart-forecasting
- **WandB (Giga):** https://wandb.ai/gbera23-free-university-of-tbilisi-/walmart-forecasting